# BigBasket Category Performance Diagnostic — Part 4
This notebook independently cleans `orders_raw.csv`, analyzes revenue, and cross-validates the top category and supplier against Part 1.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("orders_raw.csv")
products = pd.read_csv("products.csv")

print("Shape:", df.shape)
df.info()
display(df.describe(include="all"))
print(df["status"].value_counts(dropna=False))


## Initial data-quality observations
The raw export contains more than 500 rows because duplicates were injected. City/category contain mixed casing or surrounding whitespace. `amount_inr` contains missing values and some unusually large values. Ratings are expected to be null for Cancelled/Pending orders and are not treated as an error.


In [ ]:
print("Duplicate order_ids:", df["order_id"].duplicated().sum())
df = df.drop_duplicates(subset="order_id", keep="first").copy()
print("Rows after de-duplication:", len(df))


In [ ]:
df["city"] = df["city"].astype("string").str.strip().str.title()
df["category"] = df["category"].astype("string").str.strip().str.title()

print("Cities:", sorted(df["city"].dropna().unique().tolist()))
print("Categories:", sorted(df["category"].dropna().unique().tolist()))


In [ ]:
df["amount_inr"] = pd.to_numeric(df["amount_inr"], errors="coerce")
missing_amount = df["amount_inr"].isna().sum()
print("Missing amount_inr:", missing_amount)

rating_null_by_status = df.groupby("status")["rating"].apply(lambda s: s.isna().sum())
print("Null ratings by status:")
print(rating_null_by_status)


Missing `amount_inr` values are excluded from every revenue calculation below rather than filled with zero or a mean. Rating nulls for Cancelled/Pending are legitimate because only Delivered orders receive ratings, so those nulls remain untouched.


In [ ]:
delivered = df.loc[(df["status"] == "Delivered") & df["amount_inr"].notna(), "amount_inr"]
q1 = delivered.quantile(0.25)
q3 = delivered.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
print({"Q1": q1, "Q3": q3, "IQR": iqr, "upper_fence": upper_fence})

outlier_mask = (df["status"] == "Delivered") & df["amount_inr"].notna() & (df["amount_inr"] > upper_fence)
print("Rows capped:", int(outlier_mask.sum()))

df.loc[df["status"] == "Delivered", "amount_inr"] = df.loc[df["status"] == "Delivered", "amount_inr"].clip(upper=upper_fence)


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()
df["revenue_per_unit"] = df["amount_inr"] / df["quantity"]
df["is_delivered"] = df["status"].eq("Delivered")


In [ ]:
category_revenue = (
    df.loc[df["is_delivered"] & df["amount_inr"].notna()]
      .groupby("category", as_index=False)["amount_inr"]
      .sum()
      .rename(columns={"amount_inr":"total_revenue"})
      .sort_values("total_revenue", ascending=False)
)
display(category_revenue)
top_category = category_revenue.iloc[0]["category"]
print("Top category by revenue:", top_category)

merged = df.merge(products[["product_id","supplier"]], on="product_id", how="left")
supplier_revenue = (
    merged.loc[merged["is_delivered"] & merged["amount_inr"].notna()]
      .groupby("supplier", as_index=False)["amount_inr"]
      .sum()
      .rename(columns={"amount_inr":"total_revenue"})
      .sort_values("total_revenue", ascending=False)
)
display(supplier_revenue)
top_supplier = supplier_revenue.iloc[0]["supplier"]
print("Top supplier by revenue:", top_supplier)

print("Matches Part 1 top category:", top_category == "Household Essentials")
print("Matches Part 1 top supplier:", top_supplier == "HomeEssentials Traders")


In [ ]:
# Chart 1: category revenue
plt.figure(figsize=(9,5))
plt.bar(category_revenue["category"], category_revenue["total_revenue"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Revenue (INR)")
plt.xlabel("Category")
plt.title(f"Household Essentials leads cleaned Delivered revenue at {category_revenue.iloc[0]['total_revenue']:.0f} INR")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 2: monthly Delivered revenue
monthly_revenue = (
    df.loc[df["is_delivered"] & df["amount_inr"].notna()]
      .groupby(df.loc[df["is_delivered"] & df["amount_inr"].notna(), "order_date"].dt.to_period("M"))["amount_inr"]
      .sum()
)
plt.figure(figsize=(9,5))
plt.plot(monthly_revenue.index.astype(str), monthly_revenue.values, marker="o")
plt.ylabel("Revenue (INR)")
plt.xlabel("Month")
plt.title("Monthly Delivered revenue trend after cleaning")
plt.tight_layout()
plt.show()


In [ ]:
# Chart 3: supplier revenue
plt.figure(figsize=(9,5))
plt.bar(supplier_revenue["supplier"], supplier_revenue["total_revenue"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Revenue (INR)")
plt.xlabel("Supplier")
plt.title(f"HomeEssentials Traders leads supplier revenue at {supplier_revenue.iloc[0]['total_revenue']:.0f} INR")
plt.tight_layout()
plt.show()


## Exactly 3 observations

### 1. What
Household Essentials is the highest-revenue category after cleaning, with the exact notebook-computed revenue shown in the category table above.
**Why it matters:** It is the largest revenue contributor in this cleaned Delivered-order view.
**Next step:** Review its product and supplier mix before allocating additional category-level effort.

### 2. What
HomeEssentials Traders is the highest-revenue supplier in the merged dataset, with the exact notebook-computed revenue shown above.
**Why it matters:** A large share of supplier-attributed revenue is concentrated with this supplier.
**Next step:** Review the supplier's category/product contribution and service metrics before making sourcing decisions.

### 3. What
The monthly Delivered-revenue chart shows the month-by-month movement after missing revenues are excluded and IQR outliers are capped.
**Why it matters:** Monthly movement identifies periods that deserve closer operational review.
**Next step:** Compare the highest- and lowest-revenue months with category-level contributions to identify the main drivers.
